<a href="https://colab.research.google.com/github/charang9/SNOW-FLAKE-PROJECTS/blob/main/Ml_P8_NEW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# WEEK 4 – DAY 18
# PROBLEM 1 – END-TO-END DATA PREPARATION & ENCODING PIPELINE
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# -----------------------------
# File 1: Customer Profile Logs
# -----------------------------
cust_profile_data = {
    'Customer_ID': [f'CUST_{200+i}' for i in range(50)],
    'Account_Tier': ['Silver','Gold','Bronze','Platinum','Gold','Silver','Bronze','Gold','Platinum','Silver',
                     'Bronze','Gold','Silver','Platinum','Bronze','Gold','Silver','Platinum','Bronze','Gold',
                     'Silver','Bronze','Gold','Platinum','Silver','Bronze','Gold','Platinum','Silver','Bronze',
                     'Gold','Silver','Platinum','Bronze','Gold','Silver','Platinum','Bronze','Gold','Silver',
                     'Bronze','Gold','Platinum','Silver','Bronze','Gold','Platinum','Silver','Bronze','Gold'],
    'Age': [34,45,np.nan,29,52,38,24,np.nan,41,31,
            27,49,36,43,np.nan,50,33,28,22,47,
            39,26,51,42,30,np.nan,46,35,25,48,
            37,32,44,23,53,40,np.nan,29,47,31,
            26,50,43,34,21,49,38,27,45,33],
    'Region': ['North','South','West','East','North','South','West','East','North','South',
               'West','East','North','South','West','East','North','South','West','East',
               'North','South','West','East','North','South','West','East','North','South',
               'West','East','North','South','West','East','North','South','West','East',
               'North','South','West','East','North','South','West','East','North','South']
}

# -----------------------------------
# File 2: Transaction History Metrics
# -----------------------------------
cust_tx_data = {
    'Customer_ID': [f'CUST_{200+i}' for i in range(50)],
    'Total_Spend_USD': [1200.5,4500.0,300.2,12500.0,3800.0,950.0,210.0,5100.0,15000.0,1100.0,
                        180.0,4200.0,1300.0,11800.0,250.0,4900.0,890.0,13200.0,190.0,4600.0,
                        1050.0,220.0,5300.0,11000.0,980.0,np.nan,4700.0,14000.0,310.0,4400.0,
                        1150.0,920.0,12100.0,150.0,5600.0,1080.0,13500.0,280.0,4800.0,1010.0,
                        230.0,5200.0,11900.0,1120.0,170.0,5000.0,12800.0,850.0,4300.0,990.0],
    'Purchase_Frequency': [12,35,2,85,28,8,3,38,92,10,
                           2,31,11,78,3,36,7,88,2,33,
                           9,3,40,75,8,1,34,90,4,32,
                           10,8,81,1,42,9,89,3,35,11,
                           2,37,80,12,1,38,86,7,31,9]
}

# ----------------------------
# File 3: Engagement Logs
# ----------------------------
cust_engagement_data = {
    'Customer_ID': [f'CUST_{200+i}' for i in range(50)],
    'App_Sessions_Per_Month': [15,42,4,95,33,11,5,48,110,14,
                               3,39,16,88,4,44,9,102,3,41,
                               12,5,52,82,10,2,43,105,6,38,
                               13,10,91,2,55,12,100,4,42,15,
                               3,46,89,14,2,47,98,8,37,11],
    'Is_High_Value': [0,1,0,1,1,0,0,1,1,0,
                      0,1,0,1,0,1,0,1,0,1,
                      0,0,1,1,0,0,1,1,0,1,
                      0,0,1,0,1,0,1,0,1,0,
                      0,1,1,0,0,1,1,0,1,0]
}

# Create DataFrames
df_profile = pd.DataFrame(cust_profile_data)
df_tx = pd.DataFrame(cust_tx_data)
df_engagement = pd.DataFrame(cust_engagement_data)

# ============================================================
# TASK 1 – Multi-Source Relational Data Join
# ============================================================
df = pd.merge(df_profile, df_tx, on="Customer_ID", how="inner")
df = pd.merge(df, df_engagement, on="Customer_ID", how="inner")

print("="*60)
print("TASK 1 – MASTER DATASET")
print("="*60)
print("Shape:", df.shape)
print(df.head())

# ============================================================
# TASK 2 – Missing Value Analysis & Group-Wise Imputation
# ============================================================
print("\n" + "="*60)
print("TASK 2 – MISSING VALUE ANALYSIS")
print("="*60)

print("\nMissing Values Before Imputation:")
print(df.isnull().sum())

age_medians = df.groupby("Account_Tier")["Age"].median()
spend_medians = df.groupby("Account_Tier")["Total_Spend_USD"].median()

print("\nGroup-Wise Medians")
for tier in age_medians.index:
    print(f"{tier:<9}: Age = {age_medians[tier]} | Spend = ${spend_medians[tier]:,.2f}")

df["Age"] = df.groupby("Account_Tier")["Age"].transform(
    lambda x: x.fillna(x.median())
)

df["Total_Spend_USD"] = df.groupby("Account_Tier")["Total_Spend_USD"].transform(
    lambda x: x.fillna(x.median())
)

print("\nMissing Values After Imputation:")
print(df.isnull().sum())

# ============================================================
# TASK 3 – Categorical Encoding
# ============================================================
print("\n" + "="*60)
print("TASK 3 – CATEGORICAL ENCODING")
print("="*60)

tier_mapping = {
    "Bronze":1,
    "Silver":2,
    "Gold":3,
    "Platinum":4
}

df["Account_Tier_Encoded"] = df["Account_Tier"].map(tier_mapping)

region_encoded = pd.get_dummies(
    df["Region"],
    prefix="Region",
    drop_first=True
)

df = pd.concat([df, region_encoded], axis=1)

print(df[[
    "Account_Tier",
    "Account_Tier_Encoded",
    "Region",
    "Region_North",
    "Region_South",
    "Region_West"
]].head())

# ============================================================
# TASK 4 – Outlier Detection & IQR Capping
# ============================================================
print("\n" + "="*60)
print("TASK 4 – OUTLIER DETECTION")
print("="*60)

Q1 = df["Total_Spend_USD"].quantile(0.25)
Q3 = df["Total_Spend_USD"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[df["Total_Spend_USD"] > upper_bound]

print(f"Q1          : ${Q1:,.2f}")
print(f"Q3          : ${Q3:,.2f}")
print(f"IQR         : ${IQR:,.2f}")
print(f"Upper Bound : ${upper_bound:,.2f}")
print(f"Outliers    : {len(outliers)}")

df["Total_Spend_USD"] = np.where(
    df["Total_Spend_USD"] > upper_bound,
    upper_bound,
    df["Total_Spend_USD"]
)

# ============================================================
# TASK 5 – Standard Feature Scaling
# ============================================================
print("\n" + "="*60)
print("TASK 5 – STANDARD SCALING")
print("="*60)

continuous_features = [
    "Age",
    "Total_Spend_USD",
    "Purchase_Frequency",
    "App_Sessions_Per_Month"
]

scaler = StandardScaler()

df[continuous_features] = scaler.fit_transform(df[continuous_features])

print("\nScaled Feature Statistics")
for col in continuous_features:
    print(f"{col:<25} Mean = {df[col].mean():.2f} | Std = {df[col].std(ddof=0):.2f}")

# ============================================================
# FINAL FEATURE MATRIX
# ============================================================
X = df.drop(columns=[
    "Customer_ID",
    "Account_Tier",
    "Region",
    "Is_High_Value"
])

y = df["Is_High_Value"]

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "="*60)
print("E-COMMERCE DATA PREPARATION & ENCODING PIPELINE")
print("="*60)

print(f"Master Dataset Records     : {len(df)}")
print("Initial Columns            : 8")
print(f"Processed Output Features  : {X.shape[1]}")

print("\nMissing Value Resolution")
print("- Age Imputations          : 5 records restored via Account_Tier median")
print("- Spend Imputations        : 1 record restored via Account_Tier median")

print("\nFeature Encoding Summary")
print("- Ordinal Encoding         : Account_Tier mapped to integer scale [1–4]")
print("- One-Hot Encoding         : Region mapped into binary columns [North, South, West]")

print("\nOutlier Handling & Feature Scaling")
print(f"- Capped Total_Spend_USD   : {len(outliers)} upper-bound outliers truncated at ${upper_bound:,.2f}")
print("- Standardization          : All continuous features scaled to Zero Mean and Unit Variance")

print("\nPipeline Outcome")
print("Clean, fully transformed feature matrix X prepared for supervised model training.")
print("="*60)

print("\nFinal Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)
print("\nFirst 5 Rows of X")
print(X.head())

TASK 1 – MASTER DATASET
Shape: (50, 8)
  Customer_ID Account_Tier   Age Region  Total_Spend_USD  Purchase_Frequency  \
0    CUST_200       Silver  34.0  North           1200.5                  12   
1    CUST_201         Gold  45.0  South           4500.0                  35   
2    CUST_202       Bronze   NaN   West            300.2                   2   
3    CUST_203     Platinum  29.0   East          12500.0                  85   
4    CUST_204         Gold  52.0  North           3800.0                  28   

   App_Sessions_Per_Month  Is_High_Value  
0                      15              0  
1                      42              1  
2                       4              0  
3                      95              1  
4                      33              1  

TASK 2 – MISSING VALUE ANALYSIS

Missing Values Before Imputation:
Customer_ID               0
Account_Tier              0
Age                       5
Region                    0
Total_Spend_USD           1
Purchase_Frequ